In [ ]:
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PyBolt import cosmology 
from PyBolt import fBE_solver as fBE
import time
import math

from scipy.interpolate import CubicSpline


from cubic_spline import TorchCubicSpline
from pinn import compute_gradients, uniform_sampler, smooth_max, smooth_abs, MS_loss_function, FCN
from BoltzmaNN import BoltzmaNN

torch.set_default_dtype(torch.float64)

In [ ]:
# The mass of the mother particle (MP)
m1=1
# The mass of the daughter particle (DP)
m2=0.5

# m1 = 1.776 # (tau) GeV
# m2 = 0.10565 # (muon) GeV

# The mass of the DM particle
mDM = 1e-10 # GeV

# PQ symmetry breaking scale
fa = 2e+8 # GeV
# corresponds to the freeze out

# Ratio of the two masses
mu = m2/m1
massRatio=m2/m1

# Number of massless particle degrees of freedom
g_x = 1.0
# Number of MP dofs
g_m1 = 2.0
# Number of DP dofs
g_m2 = 2.0

# Coupling of MP-DP-x
yMDx = 1

# Decay width of MP -> DP + x
Gamma = yMDx**2*m1**3*(1-(m2/m1)**2)**3/64/np.pi/fa**2 # GeV
# Msquared for the decay

Msq= yMDx**2*m1**4*(1 - (m2/m1)**2)**2/4/fa**2 # GeV^2

In [ ]:
# Create a model of axion phase-space evolution
AxionModel = fBE.Model(m1,mDM,g_x,'b')
# m1 is the mass of the heaviest particle in the reaction

# Create an instance of the DecayToX process class
Decay = fBE.DecayToX(m1,m2,g_m1,Msq,Gamma)

# Create the grid of x and q
N_x = 500 # number of x points
N_q = 200 # number of q points
x0 = 6.0  # starting x value
xf = 20.0 # final x value

q0 = 0.0001 # smallest q value
qf = 20. # biggest q value

# Creating vectors of linearly distributed values of x and q
x_lin = np.linspace(x0,xf,N_x)
q_lin = np.linspace(q0,qf,N_q)

# Assigning the grid (of x and q) to the model
AxionModel.changeGrid(x_lin,q_lin)

# Add the decay collision integral to the model collision term (the only process that we consider)
AxionModel.addCollisionTerm(Decay.collisionTerm)

# Initial distribution
#f0 = np.zeros_like(q_lin)
f0 = AxionModel.equilibriumFunction(q_lin)

# fBE solver options
solver_options = {
    'method': 'BDF', # Implicit backward differentiation formula method of variable order (1-5) - because the equation is stiff
    'atol': 1e-5,   # Absolute tolerance
    'rtol': 1e-3   # Relative tolerance
}

# Solving the fBE
AxionModel.solve_fBE(f0,solver_options)

NumericalFinalDistribution = AxionModel.getSolution()[-1, :]

# Plot the initial and final solutions and the equilibrium distribution to compare
plt.figure()
plt.plot(q_lin,f0, label = 'Initial')
plt.plot(q_lin, AxionModel.getSolution()[-1,:], label = 'Final')
plt.plot(q_lin, AxionModel.equilibriumFunction(q_lin),'r--', label = 'Equilibrium')
plt.xlabel(r'$q$')
plt.ylabel(r'$q^2 f(q)$')
plt.title("Final distribution function")
plt.legend(loc="upper right")
plt.grid()
plt.show()


In [ ]:
# Initialize the TrainingBoltzmaNN instance with the specified parameters
train1 = BoltzmaNN(
    hidden_width=32,
    batch_size=25,
    num_epochs=4001,
    collocation_number_plot=N_q
)

# Train the network and measure the training time
start_time = time.time()
mse_nn_num, final_loss = train1.train(
    physics_weight=10,
    positivity_weight=0,
    bc_weight=1e-1,
    initialC_weight=1,
    patience=1000,
    learning_rate=0.01
    )
end_time = time.time()
print('Learning time:', end_time - start_time, "\n\n")

# Plot the steps and learning data
# train1.plot_steps()
train1.plot_last(1)
train1.plot_learning_data()